# 🌊 AquaYOLO: AI-Based Underwater Marine Organism Detection & Classification

AquaYOLO is a deep learning system specifically designed to detect and classify marine species and humans in underwater images and videos, even in murky, low-visibility, and color-attenuated environments.

### 🔬 Complete End-to-End Pipeline in This Notebook:
1. **⚡ Fast Direct In-Colab Download**: Ultra-fast (100+ MB/s) downloads from Kaggle & Roboflow directly into Colab.
2. **🔄 Marine Dataset Harmonizer**: Zero-collision remapping of annotations into a unified 26-class marine taxonomy.
3. **🌊 Stage-1 Underwater Enhancement**: LAB CLAHE contrast enhancement and red-spectrum attenuation compensation.
4. **🧠 Custom AquaYOLO Architecture Training**:
   - CSPDarknet Backbone + SPPF
   - Multi-Scale Aqua-Attn: **Spectral Attenuation Compensator (SAC)** + **Turbidity Contrast Gate (TCG)** at P3, P4, and P5
   - PANet Multi-Scale Neck + Decoupled Anchor-Free Detection Head
   - Class-Weighted Focal Loss
   - Cosine Annealing LR Scheduler with Warmup + Underwater Augmentation
5. **📊 Comprehensive Evaluation**: Per-class AP, mAP@0.50, mAP@0.50:0.95, Precision, Recall, and F1.
6. **🔍 EigenCAM Visual Explanations**: Visual heatmaps explaining SAC spectral compensation and TCG turbidity focus.
7. **📦 One-Click Zip & Download**: Automatically bundles all `.pt` model weights, metric graphs, and logs to your local computer.

## 🚀 Step 1: Check GPU Runtime
Verify that your GPU runtime is active (Runtime -> Change runtime type -> T4 GPU).

In [ ]:
!nvidia-smi

## 📦 Step 2: Clone AquaYOLO Repository & Install Dependencies

In [ ]:
# Clone repository
!git clone https://github.com/Aravind8833/AquaYOLO.git
%cd /content/AquaYOLO

# Add src to Python path
import sys
import os
if "/content/AquaYOLO/src" not in sys.path:
    sys.path.append("/content/AquaYOLO/src")
os.environ["PYTHONPATH"] = "/content/AquaYOLO/src:" + os.environ.get("PYTHONPATH", "")

# Install required libraries
!pip install -q ultralytics opencv-python matplotlib seaborn pyyaml pandas pillow tqdm kagglehub roboflow
print('\n✅ Environment ready and AquaYOLO module loaded!')

## ⚡ Step 3: Direct Fast Download from Kaggle & Roboflow

Downloads datasets directly into Colab in under 1-2 minutes at 100+ MB/s.

### 🔑 Kaggle Authentication:
Enter your Kaggle credentials below (from your downloaded `kaggle.json`):

In [ ]:
import os
from pathlib import Path
import kagglehub

# =====================================================================
# ⚙️ OPTION 1: KAGGLE CREDENTIALS (ENTER USERNAME & KEY)
# =====================================================================
os.environ["KAGGLE_USERNAME"] = "your_kaggle_username"
os.environ["KAGGLE_KEY"] = "your_kaggle_api_key"

# Or upload kaggle.json directly if username is not filled in:
if not os.path.exists(os.path.expanduser("~/.kaggle/kaggle.json")) and os.environ["KAGGLE_USERNAME"] == "your_kaggle_username":
    print("Upload your kaggle.json file:")
    from google.colab import files
    uploaded = files.upload()
    if uploaded:
        os.makedirs(os.path.expanduser("~/.kaggle"), exist_ok=True)
        for fn in uploaded:
            os.rename(fn, os.path.expanduser(f"~/.kaggle/{fn}"))
        os.chmod(os.path.expanduser("~/.kaggle/kaggle.json"), 0o600)

# 1. Download Kaggle Dataset 1: aquarium-data-cots
print("\n1. Downloading slavkoprytula/aquarium-data-cots from Kaggle...")
path_cots = kagglehub.dataset_download("slavkoprytula/aquarium-data-cots")
print(f"   COTS downloaded to: {path_cots}")

# 2. Download Kaggle Dataset 2: sea-animals-image-dataste
print("\n2. Downloading vencerlanz09/sea-animals-image-dataste from Kaggle...")
path_sea = kagglehub.dataset_download("vencerlanz09/sea-animals-image-dataste")
print(f"   Sea Animals downloaded to: {path_sea}")

### 🐠 Download Roboflow Dataset (`marine-life-pzpu1`)
Paste your Roboflow API key or curl download URL below (optional):

In [ ]:
import subprocess
from pathlib import Path

ROBOFLOW_DOWNLOAD_DIR = Path("/content/downloads/marine_life_roboflow")
ROBOFLOW_DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)

# =====================================================================
# ⚙️ ROBOFLOW CREDENTIALS (PASTE API KEY OR CURL URL)
# =====================================================================
ROBOFLOW_API_KEY = ""   # e.g. "abc123xyz"
ROBOFLOW_CURL_URL = ""  # e.g. "https://universe.roboflow.com/ds/..."
# =====================================================================

path_roboflow = None
if ROBOFLOW_API_KEY:
    from roboflow import Roboflow
    print("Downloading from Roboflow via Python API...")
    rf = Roboflow(api_key=ROBOFLOW_API_KEY)
    project = rf.workspace("boats-hr2s2").project("marine-life-pzpu1")
    dataset = project.version(1).download("yolov8", location=str(ROBOFLOW_DOWNLOAD_DIR))
    path_roboflow = ROBOFLOW_DOWNLOAD_DIR
    print(f"✅ Roboflow dataset downloaded to: {path_roboflow}")
elif ROBOFLOW_CURL_URL:
    print("Downloading from Roboflow via curl...")
    zip_dest = ROBOFLOW_DOWNLOAD_DIR / "roboflow.zip"
    subprocess.run(f"curl -L '{ROBOFLOW_CURL_URL}' > '{zip_dest}'", shell=True, check=True)
    subprocess.run(f"unzip -q '{zip_dest}' -d '{ROBOFLOW_DOWNLOAD_DIR}'", shell=True, check=True)
    path_roboflow = ROBOFLOW_DOWNLOAD_DIR
    print(f"✅ Roboflow dataset downloaded and unzipped to: {path_roboflow}")
else:
    print("ℹ️ Note: Proceeding with Kaggle marine datasets.")

## 🔄 Step 4: Harmonize & Merge Datasets (Zero Class Collisions)
Builds `/content/dataset` with clean train, val, and test splits and remapped classes.

In [ ]:
from dataset_merger import merge_datasets

downloaded_sources = [p for p in [path_cots, path_sea, path_roboflow] if p is not None and Path(p).exists()]
print(f"Datasets ready for harmonization: {len(downloaded_sources)}")
for s in downloaded_sources:
    print(f"  📂 {Path(s).name}")

LOCAL_DATASET_DIR = Path("/content/dataset")
TEMP_EXTRACT_DIR = Path("/content/temp_extract")

DATA_YAML_PATH = merge_datasets(downloaded_sources, LOCAL_DATASET_DIR, TEMP_EXTRACT_DIR)
print(f"\n✅ Unified dataset ready: {DATA_YAML_PATH}")

## 🧠 Step 5: Train Custom AquaYOLO Architecture (Primary Model)

Trains the custom **AquaYOLO** deep learning model featuring:
- **CSPDarknet Backbone + SPPF**
- **Multi-Scale Aqua-Attn** (SAC + TCG at P3, P4, and P5)
- **PANet Feature Fusion Neck**
- **Decoupled Anchor-Free Detection Head**
- **Class-Weighted Focal Loss**
- **Cosine LR Scheduler + Warmup + Data Augmentation**

In [ ]:
# Set output directory
AQUAYOLO_OUT = "/content/AquaYOLO/runs/aquayolo_custom"

# Run Custom AquaYOLO training (Accelerated)
!python src/aquayolo/train.py \
    --data /content/dataset \
    --epochs 25 \
    --batch 32 \
    --size 640 \
    --lr 0.002 \
    --warmup-epochs 2 \
    --workers 4 \
    --val-samples 300 \
    --out "{AQUAYOLO_OUT}"

## 📊 Step 6: Comprehensive Model Evaluation (mAP, Precision, Recall)
Evaluates `best.pt` on the held-out test split, computing per-class AP, mAP@0.50, and generating evaluation graphs.

In [ ]:
EVAL_OUT = "/content/AquaYOLO/runs/aquayolo_custom/evaluation"
BEST_WEIGHTS = "/content/AquaYOLO/runs/aquayolo_custom/best.pt"

!python src/aquayolo/evaluate.py \
    --data /content/dataset \
    --weights "{BEST_WEIGHTS}" \
    --split test \
    --batch 8 \
    --out "{EVAL_OUT}"

## 🔍 Step 7: EigenCAM Visual Explanations (SAC + TCG + Attention Heatmaps)
Generates EigenCAM visual explanations highlighting the exact image regions influencing the AquaYOLO predictions.

In [ ]:
EIGENCAM_OUT = "/content/AquaYOLO/runs/aquayolo_custom/eigencam"

!python src/aquayolo/eigencam.py \
    --weights "{BEST_WEIGHTS}" \
    --image /content/dataset/images/test \
    --out "{EIGENCAM_OUT}"

## 📈 Step 8: Display All Generated Metric Graphs Inline
Displays all training loss curves, mAP progression, per-class accuracy, and EigenCAM explanations directly in Colab.

In [ ]:
from IPython.display import Image, display
from pathlib import Path
import glob

def show_image(path_str, title):
    p = Path(path_str)
    if p.exists():
        print("=" * 75)
        print(f"📊 {title}: {p.name}")
        print("=" * 75)
        display(Image(filename=str(p), width=900))
    else:
        print(f"ℹ️ {title} ({p.name}) not found yet.")

# 1. Training & Validation Results (Losses, mAP@0.50, Precision, Recall)
show_image("/content/AquaYOLO/runs/aquayolo_custom/results.png", "TRAINING & VALIDATION METRICS OVER EPOCHS")

# 2. Loss Curves & Component Breakdown
show_image("/content/AquaYOLO/runs/aquayolo_custom/loss_curve.png", "TOTAL LOSS CURVES")
show_image("/content/AquaYOLO/runs/aquayolo_custom/loss_components.png", "LOSS COMPONENTS (BOX, OBJ, CLASS-WEIGHTED FOCAL LOSS)")

# 3. Per-Class Average Precision (AP@0.50)
show_image("/content/AquaYOLO/runs/aquayolo_custom/evaluation/per_class_ap.png", "PER-CLASS AVERAGE PRECISION (AP50)")
show_image("/content/AquaYOLO/runs/aquayolo_custom/evaluation/evaluation_summary.png", "EVALUATION SUMMARY DASHBOARD")

# 4. EigenCAM Visual Explanations (SAC, TCG, Attention Heatmaps)
eigencam_plots = sorted(glob.glob("/content/AquaYOLO/runs/aquayolo_custom/eigencam/*.png"))
if eigencam_plots:
    print("\n" + "=" * 75)
    print("🔍 EIGENCAM ATTENTION HEATMAPS (SAC + TCG + AQUA-ATTN)")
    print("=" * 75)
    for ec_img in eigencam_plots[:3]:
        display(Image(filename=ec_img, width=950))


## 📦 Step 9: Download All Weights & Results in a Zip File
Packages all `.pt` model weights (`best.pt`, `last.pt`, `FishInv.pt`, `MegaFauna.pt`), metric graphs, and CSV logs into a zip file and triggers a direct browser download to your local PC.

In [ ]:
import os
import shutil
from google.colab import files
from datetime import datetime

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
zip_filename = f"AquaYOLO_All_Weights_and_Runs_{timestamp}"
zip_path = f"/content/{zip_filename}.zip"

export_dir = "/content/AquaYOLO_export_bundle"
if os.path.exists(export_dir):
    shutil.rmtree(export_dir)
os.makedirs(export_dir, exist_ok=True)

# 1. Copy all training & evaluation runs (best.pt, last.pt, results.png, curves, logs)
if os.path.exists("/content/AquaYOLO/runs"):
    shutil.copytree("/content/AquaYOLO/runs", f"{export_dir}/runs", dirs_exist_ok=True)

# 2. Copy root .pt weights (MegaFauna.pt, FishInv.pt, yolov8s.pt, etc.)
for pt_file in ["FishInv.pt", "MegaFauna.pt", "yolov8s.pt"]:
    pt_path = os.path.join("/content/AquaYOLO", pt_file)
    if os.path.exists(pt_path):
        shutil.copy2(pt_path, export_dir)

# 3. Create zip archive
print(f"Creating zip archive: {zip_path} ...")
shutil.make_archive(f"/content/{zip_filename}", "zip", export_dir)

# 4. Trigger browser download
print("✅ Archive created successfully! Triggering browser download to your PC...")
files.download(zip_path)
